# Bank Marketing Campaign - Term Deposit Prediction
## MLOps Individual Assignment

**Business Problem:** A leading bank runs marketing campaigns to offer term deposits. The goal is to predict which clients are most likely to subscribe, thereby reducing costs and increasing conversion rates.

**Dataset:** Bank Marketing Dataset — 4,119 rows × 21 features  
**Target Variable:** `y` (yes/no — subscribed to term deposit)

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Data Loading

In [ ]:
df = pd.read_csv("data_raw/bank-additional.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset shape: (4119, 21)

Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


## 3. Exploratory Data Analysis (EDA)

### 3.1 Data Types & Summary Statistics

In [3]:
df.dtypes

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

In [4]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


### 3.2 Target Variable Distribution

In [5]:
print("Target Variable Distribution:")
print(df['y'].value_counts())
print(f"\nClass Imbalance Ratio: {df['y'].value_counts()['no'] / df['y'].value_counts()['yes']:.1f}:1")
print("\nThe target is highly imbalanced (89% No vs 11% Yes).")
print("We will use class_weight='balanced' in our model to handle this.")

Target Variable Distribution:
y
no     3668
yes     451
Name: count, dtype: int64

Class Imbalance Ratio: 8.1:1

The target is highly imbalanced (89% No vs 11% Yes).
We will use class_weight='balanced' in our model to handle this.


### 3.3 Missing & Unknown Values

In [6]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nNo null values found. However, some categorical columns contain 'unknown' values:")
print()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    unknown_count = (df[col] == 'unknown').sum()
    if unknown_count > 0:
        print(f"  {col}: {unknown_count} unknowns ({unknown_count/len(df)*100:.1f}%)")
print("\n'unknown' values are treated as a separate category (handled by OneHotEncoder).")

Missing values per column:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

No null values found. However, some categorical columns contain 'unknown' values:

  job: 39 unknowns (0.9%)
  marital: 11 unknowns (0.3%)
  education: 167 unknowns (4.1%)
  default: 803 unknowns (19.5%)
  housing: 105 unknowns (2.5%)
  loan: 105 unknowns (2.5%)

'unknown' values are treated as a separate category (handled by OneHotEncoder).


## 4. Data Preprocessing

### 4.1 Encode Target Variable

In [7]:
le = LabelEncoder()
df['y_encoded'] = le.fit_transform(df['y'])  # no=0, yes=1
print(f"Target encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Target encoding: {'no': np.int64(0), 'yes': np.int64(1)}


### 4.2 Feature-Target Separation

In [9]:
X = df.drop(columns=['y', 'y_encoded'])
y = df['y_encoded']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print("In production, duration causes data leakage and should be removed.")

Numeric features (10): ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical features (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
In production, duration causes data leakage and should be removed.


### 4.3 Preprocessing Pipeline

We use a `ColumnTransformer` to:
- **StandardScaler** for numeric features
- **OneHotEncoder** for categorical features (handles 'unknown' as a category)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)
print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 5. Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

Training set: 3295 samples
Test set: 824 samples

Training target distribution:
y_encoded
0    2934
1     361
Name: count, dtype: int64

Test target distribution:
y_encoded
0    734
1     90
Name: count, dtype: int64


## 6. Model Training — Random Forest Classifier

We use `class_weight='balanced'` to handle the 8:1 class imbalance ratio.

In [12]:
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)
print("Model training completed successfully!")

Model training completed successfully!


## 7. Model Evaluation

In [13]:
y_pred = model_pipeline.predict(X_test)
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

MODEL PERFORMANCE METRICS
Accuracy:  0.8896
Precision: 0.4962
Recall:    0.7222
F1-Score:  0.5882
ROC-AUC:   0.9425


### 7.1 Classification Report

In [14]:
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

              precision    recall  f1-score   support

      No (0)       0.96      0.91      0.94       734
     Yes (1)       0.50      0.72      0.59        90

    accuracy                           0.89       824
   macro avg       0.73      0.82      0.76       824
weighted avg       0.91      0.89      0.90       824



### 7.2 Confusion Matrix

In [15]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"                 Predicted No  Predicted Yes")
print(f"  Actual No      {cm[0][0]:>10}  {cm[0][1]:>13}")
print(f"  Actual Yes     {cm[1][0]:>10}  {cm[1][1]:>13}")

Confusion Matrix:
                 Predicted No  Predicted Yes
  Actual No             668             66
  Actual Yes             25             65


In [17]:
print(cm)

[[668  66]
 [ 25  65]]


### 7.3 Feature Importances (Top 15)

In [16]:
feature_names = (numeric_features +
                 list(model_pipeline.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_features)))
importances = model_pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(15)

print("Top 15 Feature Importances:")
print("-" * 45)
for _, row in feat_imp.iterrows():
    bar = "█" * int(row['importance'] * 100)
    print(f"  {row['feature']:30s} {row['importance']:.4f} {bar}")

Top 15 Feature Importances:
---------------------------------------------
  duration                       0.3547 ███████████████████████████████████
  euribor3m                      0.0996 █████████
  nr.employed                    0.0833 ████████
  emp.var.rate                   0.0559 █████
  age                            0.0422 ████
  cons.conf.idx                  0.0378 ███
  cons.price.idx                 0.0308 ███
  campaign                       0.0240 ██
  pdays                          0.0194 █
  poutcome_nonexistent           0.0135 █
  previous                       0.0131 █
  poutcome_success               0.0119 █
  contact_telephone              0.0110 █
  contact_cellular               0.0098 
  month_may                      0.0085 


## 8. Save Model

Save the complete pipeline (preprocessor + model) so the API can accept raw feature values directly.

In [18]:
joblib.dump(model_pipeline, 'model.pkl')
print("Model saved as 'model.pkl'")

# Save metadata
metadata = {
    'model_type': 'RandomForestClassifier',
    'features': list(X.columns),
    'metrics': {
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'roc_auc': round(roc_auc, 4)
    }
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Model metadata saved as 'model_metadata.json'")
print("\nAll files ready for deployment!")

Model saved as 'model.pkl'
Model metadata saved as 'model_metadata.json'

All files ready for deployment!


## Summary

| Step | Status |
|------|--------|
| Data Loading | ✅ Complete |
| EDA | ✅ Complete |
| Preprocessing | ✅ Complete |
| Model Training | ✅ Complete |
| Evaluation | ✅ Complete |
| Model Saving | ✅ Complete |

**Next Steps:** Deploy using Docker → AWS EC2 → Test inference via the API endpoint.